# 02 · 추론 — YuNet + Eye/Yawn CNN

학습된 가중치를 **불러와서 쓰기만** 합니다. 학습은 `01_TRAIN.ipynb` 에서 이미 끝났습니다.

## 필요한 것

```
model/artifacts/eye_model.keras          <- 01_TRAIN.ipynb 이 만듦
model/artifacts/yawn_model.keras         <- 01_TRAIN.ipynb 이 만듦
data/models/face_detection_yunet_2023mar.onnx   <- 사전학습 (학습 안 함)
```

**데이터셋은 없어도 동작합니다** — 샘플 이미지 테스트 셀만 건너뜁니다.

## 실행 순서

1~4번(셀 6개)만 돌리면 추론 준비가 끝납니다. 그 뒤는 원하는 것만 실행하세요.

| 섹션 | 내용 | 비고 |
|---|---|---|
| 1~4 | 환경 · 경로 · detector 로딩 | **필수** |
| 5 | 데이터셋 이미지로 테스트 | 웹캠 없이 확인 |
| 6 | 카메라 점검 | 웹캠 쓸 때 |
| 7 | 사진 한 장 촬영 | 수동 실행 |
| 8 | **실시간 점수 모니터** | 수동 실행 |

> ⚠ **7·8번은 카메라를 열고 대기하므로 `Run All` 에 적합하지 않습니다.**
> 6번까지 Run All 한 뒤 7·8은 직접 실행하세요.

> 커널: `C:\Users\psh03\AppData\Local\Programs\Python\Python312\python.exe`

## 1. 환경 점검

In [ ]:
import sys, platform
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("실행파일:", sys.executable)

import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

print("\nTensorFlow :", tf.__version__)
print("OpenCV     :", cv2.__version__)
print("FaceDetectorYN (YuNet):", "OK" if hasattr(cv2, "FaceDetectorYN") else "없음 (OpenCV 4.5.4+ 필요)")

## 2. 경로 · 가중치 확인

가중치가 없으면 여기서 멈추고 무엇을 해야 하는지 알려줍니다.

In [ ]:
import sys
from pathlib import Path

# =====================================================================
# 경로는 저장소 최상단 config.py 에서 가져온다. (README §10)
#
# 노트북에는 __file__ 이 없어서 config.py 의 "위치"만 cwd 기준으로 찾는다.
# 하지만 그 뒤의 모든 경로는 config.py 가 자신의 __file__ 로 계산하므로,
# 노트북을 어느 폴더에서 열든 결과가 같다. 못 찾으면 조용히 넘어가지 않고
# 즉시 멈춘다.
# =====================================================================
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "config.py").exists()), None)
if _root is None:
    raise FileNotFoundError(
        f"config.py 를 찾지 못했습니다 (탐색 시작: {_here}).\n"
        "저장소를 clone 한 폴더 안에서 노트북을 열었는지 확인하세요."
    )
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import config

PROJECT_ROOT = config.PROJECT_ROOT

ARTIFACT_DIR    = config.ARTIFACT_DIR
EYE_MODEL_PATH  = ARTIFACT_DIR / "eye_model.keras"
YAWN_MODEL_PATH = ARTIFACT_DIR / "yawn_model.keras"
YUNET_MODEL     = config.YUNET_MODEL

# 데이터셋은 추론에 꼭 필요하지 않다 (샘플 이미지 테스트용)
TEST_DIR    = config.DATA_DIR / "raw" / "dataset_new" / "test"
HAS_DATASET = TEST_DIR.exists()

RELEASE_URL = "https://github.com/lcsvvo/Driver-Drowsiness-Detection/releases"

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print()

missing = []
for label, p in [("Eye CNN  ", EYE_MODEL_PATH),
                 ("Yawn CNN ", YAWN_MODEL_PATH),
                 ("YuNet    ", YUNET_MODEL)]:
    if p.exists():
        print(f"  [o] {label} {p.name:34s} {p.stat().st_size/1024/1024:7.2f} MB")
    else:
        print(f"  [X] {label} {p.name:34s} 없음")
        missing.append((label.strip(), p))

print(f"  {'[o]' if HAS_DATASET else '[-]'} 데이터셋   {TEST_DIR.relative_to(PROJECT_ROOT)}")

if missing:
    print("\n" + "=" * 62)
    for label, p in missing:
        if "CNN" in label:
            print(f"[!] {label} 가중치가 없습니다: {p.relative_to(PROJECT_ROOT)}")
            print("    -> Release 에서 받아 model/artifacts/ 에 넣으세요:")
            print(f"       {RELEASE_URL}")
            print("    -> 직접 학습하려면 01_TRAIN.ipynb 를 실행하세요.")
        else:
            print(f"[!] YuNet 가중치가 없습니다: {p.relative_to(PROJECT_ROOT)}")
            print("    -> 저장소에 포함된 파일입니다. git pull 을 먼저 해보세요.")
            print("       https://github.com/opencv/opencv_zoo/tree/main/models/face_detection_yunet")
    print("=" * 62)
    raise FileNotFoundError("필요한 가중치가 없습니다. 위 안내를 따라주세요.")

print("\n필요한 파일이 모두 준비됐습니다.")


## 3. Detector 정의 (YuNet)

`detect_face()` 가 YuNet 출력을 MTCNN 과 같은 dict 모양으로 변환하므로,
`extract_eyes` / `extract_mouth` / CNN 추론부는 MTCNN 판과 동일합니다.

```python
face = {
    "box": [x, y, w, h],
    "confidence": score,
    "keypoints": {"left_eye":…, "right_eye":…, "nose":…,
                  "mouth_left":…, "mouth_right":…},
}
```

### 좌/우 명명이 서로 반대입니다

같은 얼굴에서 두 검출기의 좌표를 비교한 결과:

```
MTCNN left_eye  (193,167)  ↔  YuNet right_eye (194,170)   거리 7px  ← 같은 점
MTCNN right_eye (273,163)  ↔  YuNet left_eye  (274,167)   거리 7px  ← 같은 점
```

MTCNN 은 **이미지상 위치**로, YuNet 은 **인물 기준**으로 이름을 붙입니다.
`_to_mtcnn_format()` 에서 교차 매핑했습니다.

### `YAWN_USE_MOUTH_CROP`

```python
False  # 프레임 전체를 하품 CNN 에 입력 (학습 데이터와 일치)  <- 현재
True   # MTCNN 판 원래 방식 (입 crop)
```

`yawn`/`no_yawn` 학습 이미지가 640×480 **얼굴 전체**라 `False` 가 학습과 맞습니다.

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt


# 하품 추론 입력 방식
#   True  : 입 crop  /  False : 프레임 전체 (원본 레포 방식)
YAWN_USE_MOUTH_CROP = False


class Drowsiness_Detector:
    """MTCNN 판과 동일한 인터페이스. 얼굴 검출만 YuNet 으로 교체."""

    def __init__(self, list_models, yunet_path=None,
                 score_threshold=0.6, nms_threshold=0.3, top_k=5000):

        self.list_models = [str(p) for p in list_models]

        # ---- YuNet 검출기 ----
        self.yunet_path = str(yunet_path or YUNET_MODEL)
        self.score_threshold = score_threshold

        self._input_size = (320, 240)      # setInputSize 로 계속 바뀐다
        self.detector = cv2.FaceDetectorYN.create(
            self.yunet_path, "", self._input_size,
            score_threshold, nms_threshold, top_k
        )

        # Eye / Yawn 모델 로드
        self.models = self.get_models_ready()

    # =========================================================
    # 1. 모델 로드
    # =========================================================
    def get_models_ready(self):
        eye_model = tf.keras.models.load_model(self.list_models[0])
        yawn_model = tf.keras.models.load_model(self.list_models[1])
        return eye_model, yawn_model

    # =========================================================
    # 2. 이미지 Sharpen (원본 유지)
    # =========================================================
    @staticmethod
    def sharpen(img):
        kernel = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])
        return cv2.filter2D(src=img, ddepth=-1, kernel=kernel)

    # =========================================================
    # 3. YuNet 출력 -> MTCNN 형식 dict
    # =========================================================
    @staticmethod
    def _to_mtcnn_format(f):
        """YuNet 의 15개 값을 MTCNN 과 같은 dict 로 변환.

        YuNet 원시 출력:
          [0:4]   x, y, w, h
          [4:6]   right_eye   (인물 기준 오른쪽 = 이미지 왼쪽)
          [6:8]   left_eye    (인물 기준 왼쪽  = 이미지 오른쪽)
          [8:10]  nose
          [10:12] mouth_right (인물 기준)
          [12:14] mouth_left  (인물 기준)
          [14]    score

        MTCNN 은 이미지상 위치로 이름을 붙이므로 좌/우를 서로 바꿔 매핑한다.
        """
        x, y, w, h = (int(v) for v in f[0:4])

        return {
            "box": [x, y, w, h],
            "confidence": float(f[14]),
            "keypoints": {
                # 교차 매핑: YuNet right_eye -> MTCNN left_eye
                "left_eye":    (int(f[4]),  int(f[5])),
                "right_eye":   (int(f[6]),  int(f[7])),
                "nose":        (int(f[8]),  int(f[9])),
                "mouth_left":  (int(f[10]), int(f[11])),
                "mouth_right": (int(f[12]), int(f[13])),
            },
        }

    # =========================================================
    # 4. 얼굴 검출 (YuNet)
    # =========================================================
    def detect_face(self, image):
        """가장 큰 얼굴 하나를 MTCNN 형식으로 반환. 없으면 None.

        YuNet 은 BGR 을 그대로 받으므로 MTCNN 판에 있던 cvtColor 가 필요 없다.
        """
        h, w = image.shape[:2]

        # 입력 크기가 바뀌면 알려줘야 한다
        if (w, h) != self._input_size:
            self.detector.setInputSize((w, h))
            self._input_size = (w, h)

        _, faces = self.detector.detect(image)

        if faces is None or len(faces) == 0:
            return None

        # 가장 큰 얼굴
        best = max(faces, key=lambda f: f[2] * f[3])

        return self._to_mtcnn_format(best)

    # =========================================================
    # 5. 양쪽 눈 영역 추출  (MTCNN 판과 완전히 동일)
    # =========================================================
    def extract_eyes(self, image, face=None):

        if face is None:
            face = self.detect_face(image)

        if face is None:
            return None

        keypoints = face["keypoints"]

        left_eye = keypoints["left_eye"]
        right_eye = keypoints["right_eye"]

        h, w = image.shape[:2]

        face_width = face["box"][2]
        eye_size = max(int(face_width * 0.22), 20)

        def crop_eye(center):
            x, y = center

            x1 = max(int(x - eye_size), 0)
            y1 = max(int(y - eye_size), 0)
            x2 = min(int(x + eye_size), w)
            y2 = min(int(y + eye_size), h)

            eye = image[y1:y2, x1:x2]

            if eye.size == 0:
                return None, None

            eye = cv2.resize(eye, (256, 256))
            eye = self.sharpen(eye)

            return eye, (x1, y1, x2, y2)

        left, left_box = crop_eye(left_eye)
        right, right_box = crop_eye(right_eye)

        if left is None or right is None:
            return None

        return left, right, left_box, right_box, face

    # =========================================================
    # 6. 입 영역 추출  (MTCNN 판과 완전히 동일)
    # =========================================================
    def extract_mouth(self, image, face=None):

        if face is None:
            face = self.detect_face(image)

        if face is None:
            return None

        keypoints = face["keypoints"]

        mouth_left = keypoints["mouth_left"]
        mouth_right = keypoints["mouth_right"]

        center_x = int((mouth_left[0] + mouth_right[0]) / 2)
        center_y = int((mouth_left[1] + mouth_right[1]) / 2)

        mouth_width = abs(mouth_right[0] - mouth_left[0])
        mouth_width = max(mouth_width, int(face["box"][2] * 0.15))

        half_width = int(mouth_width * 1.5)
        half_height = int(mouth_width * 1.2)

        h, w = image.shape[:2]

        x1 = max(center_x - half_width, 0)
        x2 = min(center_x + half_width, w)
        y1 = max(center_y - half_height, 0)
        y2 = min(center_y + half_height, h)

        mouth = image[y1:y2, x1:x2]

        if mouth.size == 0:
            return None

        mouth = cv2.resize(mouth, (256, 256))

        return mouth, (x1, y1, x2, y2)

    # =========================================================
    # 7. Eye CNN 추론  (MTCNN 판과 동일)
    # =========================================================
    def eye_detection(self, left, right):

        left = cv2.cvtColor(left, cv2.COLOR_BGR2RGB)
        right = cv2.cvtColor(right, cv2.COLOR_BGR2RGB)

        # 모델 안에 Rescaling(1/255) 이 있으므로 0~255 그대로 입력
        inputs = np.stack([left, right]).astype(np.float32)

        predictions = self.models[0].predict(inputs, verbose=0)

        # class 0 = Closed, class 1 = Open
        left_class = np.argmax(predictions[0])
        right_class = np.argmax(predictions[1])

        closed = (left_class == 0 or right_class == 0)

        return closed, predictions

    # =========================================================
    # 8. Yawn CNN 추론  (MTCNN 판과 동일)
    # =========================================================
    def yawn_detection(self, image, face=None):

        if YAWN_USE_MOUTH_CROP:
            result = self.extract_mouth(image, face=face)

            if result is None:
                return False, None, None

            target, mouth_box = result
        else:
            target = cv2.resize(image, (256, 256))
            mouth_box = None

        target_rgb = cv2.cvtColor(target, cv2.COLOR_BGR2RGB)
        inputs = np.expand_dims(target_rgb, axis=0).astype(np.float32)

        prediction = self.models[1].predict(inputs, verbose=0)[0]

        # class 0 = yawn, class 1 = no_yawn
        yawn = (np.argmax(prediction) == 0)

        return yawn, prediction, mouth_box

    # =========================================================
    # 9. 최종 예측  (MTCNN 판과 동일)
    # =========================================================
    def predict(self, image_path, show=True):

        image = cv2.imread(str(image_path))

        if image is None:
            print("이미지를 불러올 수 없습니다:", image_path)
            return None

        face = self.detect_face(image)

        if face is None:
            print("얼굴을 찾지 못했습니다.")
            return None

        eye_result = self.extract_eyes(image, face=face)

        if eye_result is None:
            print("눈을 찾지 못했습니다.")
            return None

        left, right, left_box, right_box, face = eye_result

        eye_closed, eye_pred = self.eye_detection(left, right)

        yawn, yawn_pred, mouth_box = self.yawn_detection(image, face=face)

        print("Face conf:", round(face["confidence"], 3))
        print("Left Eye :", eye_pred[0])
        print("Right Eye:", eye_pred[1])

        if yawn_pred is not None:
            print("Yawn     :", yawn_pred)
        else:
            print("Yawn     : Mouth detection failed")

        print()
        print("[!] Eyes Closed" if eye_closed else "[o] Eyes Open")
        print("[!] Yawn Detected" if yawn else "[o] No Yawn")

        status = "DROWSINESS WARNING" if (eye_closed or yawn) else "NORMAL"
        print("\nStatus:", status)

        if show:
            vis = image.copy()

            for box in [left_box, right_box]:
                x1, y1, x2, y2 = box
                cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)

            if mouth_box is not None:
                x1, y1, x2, y2 = mouth_box
                cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)

            x, y, fw, fh = face["box"]
            cv2.rectangle(vis, (x, y), (x + fw, y + fh), (255, 0, 0), 2)

            plt.figure(figsize=(10, 7))
            plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            plt.axis("off")
            plt.title(status)
            plt.show()

        return {
            "eye_closed": eye_closed,
            "yawn": yawn,
            "status": status,
            "eye_predictions": eye_pred,
            "yawn_prediction": yawn_pred,
            "mouth_box": mouth_box,
        }


print("Drowsiness_Detector (YuNet) 정의 완료 — YAWN_USE_MOUTH_CROP =", YAWN_USE_MOUTH_CROP)

## 4. Detector 로딩

In [ ]:
detector = Drowsiness_Detector(
    [EYE_MODEL_PATH, YAWN_MODEL_PATH],
    yunet_path=YUNET_MODEL,
)

print("YuNet + 두 CNN 로딩 완료!\n")

# 검출 속도 확인 (데이터셋 이미지 또는 합성 이미지)
import time

if HAS_DATASET:
    _img = cv2.imread(str(sorted((TEST_DIR / "yawn").glob("*"))[0]))
else:
    _img = np.zeros((480, 640, 3), np.uint8)

for _ in range(3):
    detector.detect_face(_img)

t0 = time.perf_counter()
for _ in range(20):
    _f = detector.detect_face(_img)
dt = (time.perf_counter() - t0) / 20

print(f"YuNet 검출 {_img.shape[1]}x{_img.shape[0]}: {dt*1000:.2f} ms  ({1/dt:.0f} FPS)")
if _f:
    print("얼굴:", _f["box"], " conf:", round(_f["confidence"], 3))

## 5. 이미지 파일로 테스트

웹캠 없이 동작을 확인합니다. (눈 crop 이미지는 얼굴 검출이 안 되므로 얼굴 사진을 씁니다)

In [ ]:
if not HAS_DATASET:
    print("데이터셋이 없어 건너뜁니다. 아래 웹캠 셀로 진행하세요.")

else:
    for cls in ["yawn", "no_yawn"]:
        sample = sorted((TEST_DIR / cls).glob("*"))[0]
        print("=" * 55)
        print(f"샘플: {cls} -> {sample.name}")
        print("=" * 55)
        detector.predict(sample)
        print()

## 6. 카메라 점검

> **`cv2.imshow` 는 노트북에서 쓰지 않습니다.**
> VSCode 노트북 커널은 GUI 이벤트 루프가 없어 창이 멈추거나 키 입력을 못 받습니다.
> 아래는 프레임을 **노트북 안에 인라인으로** 갱신합니다.

백엔드별 읽기 속도를 직접 재서 가장 빠른 조합을 고릅니다.
어두운 곳에서 자동 노출이 셔터를 길게 잡으면 카메라가 10 FPS 까지 떨어지는데,
그 경우를 잡아내기 위해 열리는지만 보지 않고 **속도를 측정**합니다.

In [ ]:
import cv2, time
import numpy as np


def measure_camera(cap, n=25, warm=12):
    """cap.read() 평균 소요시간(초). 실패하면 None."""
    for _ in range(warm):
        cap.read()
    ts = []
    for _ in range(n):
        t = time.perf_counter()
        ok, f = cap.read()
        ts.append(time.perf_counter() - t)
        if not ok:
            return None
    return float(np.mean(ts))


def check_cameras(max_index=2):
    """백엔드 x 인덱스를 시험하고 '읽기 속도까지' 재서 가장 빠른 조합을 고른다.

    Windows 에서 DSHOW + 자동노출 조합은 어두운 환경에서 프레임레이트가
    1/3 로 떨어지는 일이 흔하다. 그래서 열리는지만 보지 않고 속도를 잰다.
    """
    backends = [("CAP_MSMF", cv2.CAP_MSMF),
                ("CAP_DSHOW", cv2.CAP_DSHOW),
                ("CAP_ANY", cv2.CAP_ANY)]
    working = []

    for bname, bid in backends:
        for idx in range(max_index):
            cap = None
            try:
                cap = cv2.VideoCapture(idx, bid)
                if not cap.isOpened():
                    continue
                dt = measure_camera(cap)
                if dt is None:
                    continue
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                print(f"  {bname:10s} index={idx}  {w}x{h}  "
                      f"{dt*1000:6.1f} ms  ({1/dt:4.1f} FPS)")
                working.append((dt, bname, bid, idx))
            except Exception as e:
                print(f"  {bname:10s} index={idx}  오류 {e}")
            finally:
                if cap is not None:
                    cap.release()

    if not working:
        print("사용 가능한 카메라가 없습니다.")
        print("  - 다른 앱(Zoom/Teams/카메라앱)이 점유 중인지 확인")
        print("  - Windows 설정 > 개인 정보 및 보안 > 카메라 접근 허용")
        return []

    working.sort()          # 가장 빠른 것이 앞으로
    return working


print("카메라 탐색 중... (백엔드별 읽기 속도까지 측정)\n")
CAMERAS = check_cameras()

if CAMERAS:
    best_dt, best_name, CAMERA_BACKEND, CAMERA_INDEX = CAMERAS[0]
    print(f"\n선택: {best_name}  index={CAMERA_INDEX}  "
          f"({1/best_dt:.1f} FPS)")

    if best_dt > 0.05:
        print("\n[!] 읽기가 느립니다 (>50ms). 자동 노출이 원인일 수 있습니다.")
        print("    _open_camera(manual_exposure=True) 로 열어보세요.")
else:
    CAMERA_INDEX, CAMERA_BACKEND = 0, cv2.CAP_ANY

In [ ]:
import os, time, threading
import cv2
from pathlib import Path
from IPython.display import display, Image


def _open_camera(camera_index=None, backend=None,
                 manual_exposure=False, exposure=-6):
    """카메라를 열어 VideoCapture 반환. 실패 시 예외.

    manual_exposure=True 면 자동 노출을 끈다. 어두운 곳에서 카메라가
    셔터를 길게 잡아 프레임레이트가 떨어지는 것을 막지만 화면은 어두워진다.
    """
    idx = CAMERA_INDEX if camera_index is None else camera_index
    bid = CAMERA_BACKEND if backend is None else backend

    cap = cv2.VideoCapture(idx, bid)

    if not cap.isOpened():
        cap.release()
        raise RuntimeError(
            f"웹캠을 열 수 없습니다 (index={idx}).\n"
            "  - 위 check_cameras() 결과를 확인하세요\n"
            "  - 다른 앱이 카메라를 점유 중인지 확인\n"
            "  - Windows 설정 > 개인 정보 및 보안 > 카메라 접근 허용"
        )

    if manual_exposure:
        cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)   # 0.25=수동, 0.75=자동
        cap.set(cv2.CAP_PROP_EXPOSURE, exposure)

    return cap


class FrameGrabber:
    """카메라를 백그라운드 스레드에서 계속 읽어 '가장 최근 프레임'만 유지한다.

    cap.read() 는 다음 프레임이 나올 때까지 블로킹한다. 메인 루프에서 직접
    호출하면 [카메라 대기 -> 추론] 이 직렬로 쌓이지만, 스레드로 분리하면
    추론하는 동안 카메라가 병렬로 프레임을 받아둬서 대기가 사라진다.
    """

    def __init__(self, cap):
        self.cap = cap
        self.lock = threading.Lock()
        self.frame = None
        self.running = True
        self.count = 0
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def _loop(self):
        while self.running:
            ok, f = self.cap.read()
            if not ok:
                self.running = False
                break
            with self.lock:
                self.frame = f
                self.count += 1

    def read(self, timeout=5.0):
        """가장 최근 프레임을 반환. 아직 없으면 잠깐 기다린다."""
        t0 = time.time()
        while True:
            with self.lock:
                if self.frame is not None:
                    return True, self.frame.copy()
            if not self.running or (time.time() - t0) > timeout:
                return False, None
            time.sleep(0.002)

    def read_new(self, last_seq, timeout=5.0):
        """직전에 처리한 것보다 '새로운' 프레임이 나올 때까지 기다렸다 반환.

        카메라는 30 FPS 인데 루프가 100 FPS 로 돌면 같은 프레임을 세 번씩
        추론하게 된다. 결과는 당연히 같으므로 순수 낭비다. 새 프레임에만
        연산을 쓰도록 시퀀스 번호로 거른다.
        """
        t0 = time.time()
        while True:
            with self.lock:
                if self.frame is not None and self.count != last_seq:
                    return True, self.frame.copy(), self.count
            if not self.running or (time.time() - t0) > timeout:
                return False, None, last_seq
            time.sleep(0.001)

    def stop(self):
        self.running = False
        self.thread.join(timeout=1.0)
        self.cap.release()


def _show_inline(frame, handle=None, quality=80):
    """BGR 프레임을 노트북에 인라인 표시. display handle 을 반환."""
    ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        return handle

    img = Image(data=buf.tobytes())

    if handle is None:
        return display(img, display_id=True)

    handle.update(img)
    return handle


def take_photo(filename="photo.jpg", camera_index=None, countdown=3, warmup=10):
    """인라인 미리보기를 보여주며 countdown 초 후 한 장 촬영."""
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)

    cap = _open_camera(camera_index)
    handle = None
    frame = None

    try:
        for _ in range(warmup):
            cap.read()

        t_end = time.time() + countdown

        while True:
            ok, frame = cap.read()
            if not ok:
                raise RuntimeError("프레임을 읽지 못했습니다.")

            remain = t_end - time.time()
            if remain <= 0:
                break

            vis = frame.copy()
            cv2.putText(vis, f"{remain:.1f}", (25, 65),
                        cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 255, 255), 4)
            cv2.putText(vis, "get ready...", (25, 105),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            handle = _show_inline(vis, handle)

    finally:
        cap.release()

    cv2.imwrite(str(filename), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
    _show_inline(frame, handle)

    print("저장:", filename)
    return filename


CAPTURE_DIR = config.OUTPUTS_DIR / "captures"
CAPTURE_DIR.mkdir(parents=True, exist_ok=True)
print("촬영 이미지 저장 위치:", CAPTURE_DIR)

## 7. 사진 한 장 촬영  *(직접 실행)*

카운트다운 후 자동 촬영합니다. 표정을 바꿔가며 여러 번 찍어보세요.

In [ ]:
photo = take_photo(CAPTURE_DIR / "test.jpg", countdown=3)

if photo:
    result = detector.predict(photo)

## 8. 실시간 점수 모니터  *(직접 실행)*

**정지: 셀 왼쪽의 ■ (interrupt) 버튼** 또는 `max_seconds` 만료.

| 항목 | 의미 |
|---|---|
| `EYE CLOSED` | 양쪽 눈 중 **감김 확률이 높은 쪽** ("한쪽이라도 감기면 Closed" 판정이므로) |
| `YAWN` | 하품 확률 (class 0) |
| `DROWSY` | `max(EYE, YAWN)` 의 EMA — 눈 깜빡임 한 번에 경고가 뜨지 않도록 시간축 누적 |
| `PERCLOS` | 최근 1분 중 **눈이 감겨 있던 시간의 비율** (아래 셀 참고) |

`DROWSY` 는 최근 몇 초에 반응하는 순간 지표, `PERCLOS` 는 1분 단위 누적 지표라
서로 다른 것을 봅니다. 짧게 확 감으면 `DROWSY` 가 먼저 뜨고,
자주 오래 감고 있으면 `PERCLOS` 가 올라갑니다.

### 프레임 예산 (실측)

| | 시간 |
|---|---|
| YuNet 검출 (640×480) | 7~10 ms |
| Eye CNN | 15.2 ms |
| Yawn CNN | 11.6 ms |
| **연산 합계** | **37.3 ms → 약 27 FPS** |

카메라가 30 FPS 라 실질 상한은 27~29 FPS 입니다.
`FrameGrabber` 가 카메라 읽기를 백그라운드 스레드로 분리하고,
시퀀스 번호로 **같은 프레임을 중복 추론하지 않도록** 거릅니다.

이제 병목은 검출기가 아니라 **직접 학습한 CNN 두 개**입니다.

### PERCLOS — 최근 1분 중 눈이 감긴 시간 비율

`EYE CLOSED` 는 그 순간의 확률이고, `DROWSY` 는 EMA(지수이동평균)라 최근 몇 초에 강하게
쏠립니다. 둘 다 "지난 1분 동안 얼마나 감고 있었나"는 답하지 못합니다. 그걸 재는 게 PERCLOS 입니다.

**프레임 수가 아니라 시간으로 셉니다.**
FPS 가 27~29 사이에서 흔들리고 NO FACE 구간에서는 더 떨어지므로, 프레임을 세면
느린 구간이 과소평가됩니다. 프레임 간 간격(`dt`)을 더합니다.

**얼굴을 못 잡은 시간은 분모에서 뺍니다.**
안 그러면 고개를 돌린 시간이 통째로 "눈 뜬 시간"으로 들어가 PERCLOS 가 낮게 나옵니다.
대신 `coverage`(창 안에서 얼굴이 잡힌 시간 비율)를 같이 보고, 너무 낮으면 값을 신뢰하지 않습니다.

```
PERCLOS  = (눈 감긴 시간) / (얼굴이 잡힌 시간)     <- 최근 window_sec 구간
coverage = (얼굴이 잡힌 시간) / (창 전체 시간)
```

| 기준 | 값 |
|---|---|
| 각성 상태 | 보통 0.05 미만 |
| 졸음 경고 | 0.15 이상 (`perclos_threshold` 기본값) |

> 창이 1분이면 **1분이 지나야** 값이 제대로 찹니다. `max_seconds=60` 으로 돌리면
> 마지막 순간에야 창이 다 차므로, PERCLOS 를 보려면 `max_seconds=180` 정도로 길게 도세요.
> 창이 `min_window_sec`(기본 10초)만큼 차기 전에는 `warming up` 으로 표시합니다.

In [ ]:
import time
from collections import deque


class PerclosTracker:
    """최근 window_sec 동안 '눈이 감긴 시간 비율'(PERCLOS)을 계산한다.

    프레임을 세지 않고 프레임 간 간격(dt)을 더한다. FPS 가 일정하지 않기 때문이다.
    얼굴을 못 잡은 구간은 분모에서 빼고, 대신 coverage 로 신뢰도를 따로 보고한다.
    """

    def __init__(self, window_sec=60.0, closed_threshold=0.5,
                 min_window_sec=10.0, max_gap=0.5):
        self.window_sec = window_sec
        self.closed_threshold = closed_threshold
        self.min_window_sec = min_window_sec

        # 셀을 멈췄다 재개하거나 카메라가 스톨하면 dt 가 수 초로 튄다.
        # 그 한 프레임이 창을 통째로 채우지 않도록 자른다.
        self.max_gap = max_gap

        self.samples = deque()        # (t, dt, closed, face_ok)
        self.prev_t = None

        # 창과 무관한 세션 전체 누적
        self.total_time = 0.0
        self.face_time = 0.0
        self.closed_time = 0.0

        # 연속으로 감고 있던 최장 시간 (microsleep 탐지용)
        self.cur_closure = 0.0
        self.max_closure = 0.0

    def update(self, eye_p, face_ok, t=None):
        """한 프레임 반영. eye_p 는 '감김' 확률(class 0)."""
        t = time.time() if t is None else t

        if self.prev_t is None:       # 첫 프레임은 dt 를 모른다
            self.prev_t = t
            return

        dt = min(t - self.prev_t, self.max_gap)
        self.prev_t = t

        closed = bool(face_ok and eye_p >= self.closed_threshold)

        self.samples.append((t, dt, closed, face_ok))
        self.total_time += dt

        if face_ok:
            self.face_time += dt
            if closed:
                self.closed_time += dt
                self.cur_closure += dt
                self.max_closure = max(self.max_closure, self.cur_closure)
            else:
                self.cur_closure = 0.0
        else:
            self.cur_closure = 0.0    # 얼굴을 놓친 구간은 연속으로 잇지 않는다

        cutoff = t - self.window_sec
        while self.samples and self.samples[0][0] < cutoff:
            self.samples.popleft()

    def window(self):
        """최근 창 기준 (perclos, coverage, 창에 쌓인 시간, 신뢰 가능 여부)."""
        span = closed_t = face_t = 0.0
        for _, dt, closed, face_ok in self.samples:
            span += dt
            if face_ok:
                face_t += dt
                if closed:
                    closed_t += dt

        perclos = closed_t / face_t if face_t > 0 else 0.0
        coverage = face_t / span if span > 0 else 0.0
        ready = span >= self.min_window_sec and coverage >= 0.3

        return perclos, coverage, span, ready

    def session(self):
        """세션 전체 기준 (perclos, coverage, 총 시간, 최장 연속 감김 시간)."""
        perclos = self.closed_time / self.face_time if self.face_time > 0 else 0.0
        coverage = self.face_time / self.total_time if self.total_time > 0 else 0.0
        return perclos, coverage, self.total_time, self.max_closure


print("PerclosTracker 정의 완료")

In [ ]:
import numpy as np
from collections import deque


def _draw_bar(img, label, value, y, color, width=260, height=18, x=15):
    cv2.rectangle(img, (x, y), (x + width, y + height), (60, 60, 60), -1)
    filled = int(width * float(np.clip(value, 0, 1)))
    cv2.rectangle(img, (x, y), (x + filled, y + height), color, -1)
    cv2.rectangle(img, (x, y), (x + width, y + height), (200, 200, 200), 1)
    cv2.putText(img, f"{label:11s} {value:0.2f}", (x + width + 10, y + height - 3),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)


def _draw_sparkline(img, hist, x, y, w, h, threshold):
    cv2.rectangle(img, (x, y), (x + w, y + h), (35, 35, 35), -1)
    cv2.rectangle(img, (x, y), (x + w, y + h), (120, 120, 120), 1)

    ty = int(y + h * (1 - threshold))
    for sx in range(x, x + w, 12):
        cv2.line(img, (sx, ty), (min(sx + 6, x + w), ty), (0, 0, 255), 1)

    if len(hist) >= 2:
        pts = []
        for i, v in enumerate(hist):
            px = x + int(w * i / (len(hist) - 1))
            py = y + int(h * (1 - float(np.clip(v, 0, 1))))
            pts.append((px, py))
        cv2.polylines(img, [np.array(pts, np.int32)], False, (0, 255, 255), 2)

    cv2.putText(img, "DROWSY history", (x + 5, y - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)


def run_realtime_scores(camera_index=None,
                        max_seconds=60,
                        detect_width=None,
                        ema_alpha=0.3,
                        threshold=0.6,
                        history_len=120,
                        mirror=True,
                        threaded=True,
                        manual_exposure=False,
                        perclos_window=60.0,
                        perclos_threshold=0.15,
                        eye_closed_threshold=0.5):
    """실시간 졸음 점수 모니터. 정지하려면 ■(interrupt) 버튼을 누르세요.

    threaded=True 면 카메라 읽기를 백그라운드 스레드로 분리해
    [카메라 대기 -> 추론] 직렬 구조를 없앤다.

    PERCLOS 는 최근 perclos_window 초 중 눈이 감겨 있던 시간의 비율이다.
    창이 다 차야 의미가 있으므로 max_seconds 를 창보다 넉넉히 잡으세요.

    반환: 세션 요약 dict.
    """

    cap = _open_camera(camera_index, manual_exposure=manual_exposure)
    grabber = FrameGrabber(cap) if threaded else None

    last_seq = -1

    def get_frame():
        """새 프레임만 가져온다 (스레드 모드). 중복 추론 방지."""
        nonlocal last_seq
        if grabber:
            ok, f, seq = grabber.read_new(last_seq)
            last_seq = seq
            return ok, f
        return cap.read()

    handle = None
    hist = deque(maxlen=history_len)
    drowsy_ema = 0.0

    perclos_tracker = PerclosTracker(window_sec=perclos_window,
                                     closed_threshold=eye_closed_threshold)
    perclos = coverage = span = 0.0
    perclos_ready = False

    eye_p = yawn_p = 0.0
    boxes = []
    face_ok = False

    t0 = time.time()
    frames = 0
    fps = 0.0

    eye_net, yawn_net = detector.models

    try:
        while True:
            if max_seconds and (time.time() - t0) > max_seconds:
                break

            ok, frame = get_frame()
            if not ok:
                break

            if mirror:
                frame = cv2.flip(frame, 1)

            frames += 1
            h0, w0 = frame.shape[:2]

            # ---- 얼굴 검출 (YuNet) ----
            if detect_width and detect_width < w0:
                scale = detect_width / w0
                small = cv2.resize(frame, (detect_width, int(h0 * scale)))
                f = detector.detect_face(small)

                if f is not None:
                    inv = 1.0 / scale
                    face = {
                        "box": [int(v * inv) for v in f["box"]],
                        "confidence": f["confidence"],
                        "keypoints": {k: (int(v[0] * inv), int(v[1] * inv))
                                      for k, v in f["keypoints"].items()},
                    }
                else:
                    face = None
            else:
                face = detector.detect_face(frame)

            if face is not None:
                eye_result = detector.extract_eyes(frame, face=face)
                face_ok = eye_result is not None
            else:
                face_ok = False

            if face_ok:
                left, right, lbox, rbox, face = eye_result

                # model(x) 직접 호출이 .predict() 보다 3배 빠르다
                inp = np.stack([
                    cv2.cvtColor(left, cv2.COLOR_BGR2RGB),
                    cv2.cvtColor(right, cv2.COLOR_BGR2RGB),
                ]).astype(np.float32)
                pe = eye_net(inp, training=False).numpy()

                # class 0 = Closed. 한쪽이라도 감기면 Closed 이므로 max
                eye_p = float(max(pe[0][0], pe[1][0]))

                mouth_box = None
                if YAWN_USE_MOUTH_CROP:
                    mres = detector.extract_mouth(frame, face=face)
                    target = mres[0] if mres else None
                    mouth_box = mres[1] if mres else None
                else:
                    target = cv2.resize(frame, (256, 256))

                if target is not None:
                    inp_y = np.expand_dims(
                        cv2.cvtColor(target, cv2.COLOR_BGR2RGB), 0).astype(np.float32)
                    py = yawn_net(inp_y, training=False).numpy()[0]
                    yawn_p = float(py[0])          # class 0 = yawn

                boxes = [(lbox, (0, 255, 0)), (rbox, (0, 255, 0))]
                if mouth_box is not None:
                    boxes.append((mouth_box, (0, 0, 255)))
                x, y, fw, fh = face["box"]
                boxes.append(((x, y, x + fw, y + fh), (255, 0, 0)))
            else:
                eye_p *= 0.7
                yawn_p *= 0.7
                boxes = []

            # ---- 시간축 누적 ----
            # EMA: 최근 몇 초에 반응하는 순간 지표
            inst = max(eye_p, yawn_p)
            drowsy_ema = ema_alpha * inst + (1 - ema_alpha) * drowsy_ema
            hist.append(drowsy_ema)

            # PERCLOS: 최근 perclos_window 초 중 눈이 감긴 시간 비율
            # (분모는 '얼굴이 잡힌 시간'. 고개 돌린 시간을 눈 뜬 시간으로 세지 않는다)
            perclos_tracker.update(eye_p, face_ok)
            perclos, coverage, span, perclos_ready = perclos_tracker.window()

            drowsy = drowsy_ema >= threshold
            perclos_alert = perclos_ready and perclos >= perclos_threshold

            # ---- 그리기 ----
            vis = frame.copy()
            for (x1, y1, x2, y2), c in boxes:
                cv2.rectangle(vis, (x1, y1), (x2, y2), c, 2)

            panel_h = 216
            overlay = vis.copy()
            cv2.rectangle(overlay, (0, h0 - panel_h), (w0, h0), (0, 0, 0), -1)
            vis = cv2.addWeighted(overlay, 0.55, vis, 0.45, 0)

            base = h0 - panel_h + 15
            _draw_bar(vis, "EYE CLOSED", eye_p, base, (0, 200, 255))
            _draw_bar(vis, "YAWN", yawn_p, base + 26, (0, 140, 255))
            _draw_bar(vis, "DROWSY", drowsy_ema, base + 52,
                      (0, 0, 255) if drowsy else (0, 220, 0))
            _draw_bar(vis, "PERCLOS", perclos, base + 78,
                      (0, 0, 255) if perclos_alert else
                      ((0, 220, 0) if perclos_ready else (120, 120, 120)))

            # 창이 차기 전과 얼굴을 자주 놓친 구간에서는 PERCLOS 를 믿을 수 없다
            if perclos_ready:
                note = f"win {span:0.0f}s  face {coverage*100:0.0f}%"
            elif span < perclos_tracker.min_window_sec:
                note = f"warming up {span:0.0f}/{perclos_tracker.min_window_sec:0.0f}s"
            else:
                note = f"low coverage {coverage*100:0.0f}%"
            cv2.putText(vis, note, (15, base + 111),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)

            _draw_sparkline(vis, hist, 15, base + 132, 260, 60, threshold)

            if frames % 5 == 0:
                fps = frames / max(time.time() - t0, 1e-6)

            status = "DROWSY!" if drowsy else ("NORMAL" if face_ok else "NO FACE")
            scolor = (0, 0, 255) if drowsy else ((0, 220, 0) if face_ok else (150, 150, 150))
            cv2.putText(vis, status, (20, 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.1, scolor, 3)
            cv2.putText(vis, f"{fps:.1f} FPS   {time.time()-t0:.0f}s",
                        (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220, 220, 220), 1)

            if perclos_alert:
                cv2.putText(vis, f"PERCLOS {perclos*100:.0f}%", (20, 105),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            handle = _show_inline(vis, handle, quality=75)

    except KeyboardInterrupt:
        print("사용자 중지")

    finally:
        if grabber:
            grabber.stop()
        else:
            cap.release()

    s_perclos, s_coverage, s_total, s_max_closure = perclos_tracker.session()

    print(f"종료 — {frames} 프레임 처리, 평균 {fps:.1f} FPS")
    print()
    print(f"세션 {s_total:.0f}s  (얼굴 검출 {s_coverage*100:.0f}%)")
    print(f"  PERCLOS       : {s_perclos*100:5.1f} %   (임계 {perclos_threshold*100:.0f}%)")
    print(f"  최장 연속 감김: {s_max_closure:5.2f} s")

    if s_coverage < 0.3:
        print("  [!] 얼굴이 잡힌 시간이 너무 짧아 PERCLOS 를 신뢰하기 어렵습니다.")
    elif s_perclos >= perclos_threshold:
        print("  [!] PERCLOS 가 임계를 넘었습니다 — 졸음 의심")

    return {
        "frames": frames,
        "fps": fps,
        "perclos": s_perclos,
        "coverage": s_coverage,
        "duration": s_total,
        "max_closure": s_max_closure,
        "tracker": perclos_tracker,
    }


print("run_realtime_scores() 준비 완료 (YuNet + PERCLOS)")

In [ ]:
# 눈을 감아 보거나 하품해 보면서 막대와 그래프가 움직이는지 확인하세요.
# 정지: 셀 왼쪽 ■ (interrupt) 버튼
#
# PERCLOS 창이 60초라 그만큼은 돌아야 값이 찹니다 (그 전에는 warming up).

summary = run_realtime_scores(max_seconds=180, threshold=0.6,
                              perclos_window=60.0, perclos_threshold=0.15)

# 더 빠르게 (검출 4.6ms): run_realtime_scores(max_seconds=180, detect_width=320)
# 어두워서 느리면      : run_realtime_scores(max_seconds=180, manual_exposure=True)

---

## 참고

### 남아 있는 이슈

YuNet 교체는 **검출 속도만** 해결합니다. 아래는 그대로입니다.

1. **`sharpen()` 이 추론에만 적용** — 눈 학습 파이프라인에는 없어 입력 분포가 약간 어긋납니다.
2. **단일 프레임 판정** — `predict()` 는 한 장으로 판단합니다.
   실시간 셀은 EMA 로 시간축 누적을 넣어 이 문제를 완화했습니다.
3. **TEST_DIR 미사용** — 학습 시 val 을 모델 선택과 성능 보고에 모두 써서 낙관 편향이 있습니다.

### 더 빠르게 하려면

검출이 7~10ms 로 줄어든 지금, 남은 37ms 중 **27ms 가 CNN 두 개**입니다.

- 입력 256×256 → 128×128 **재학습** — 눈 crop 원본이 85~300px 라 256 은 업샘플링입니다.
- `.predict()` → `model(x, training=False)` — 실측 44.7ms → 14.8ms (실시간 셀에는 적용됨)